# Annotation in Scanpy

In [1]:
# This cell is labelled 'paramters' to work with papermill remotely
# Papermill will overwite the default local plate value below with whatever is passed to 
# the -p flag in the snakerule shell script
#os.system("conda activate eqtl_study") use this locally if using VScode
plate = 'plate1'
plate = globals().get("plate")
print(f"Processing plate: {plate}")

Processing plate: plate1


### Set Variables

In [ ]:
# Import custom utility packages, lists and functions
import sys
import os
if os.path.exists('/scratch/'):
    root_dir = '/scratch/c.c1477909/eQTL_study_2025/'
else:
    root_dir = '/Users/darren/Desktop/eQTL_study_2025/'
        
sys.path.append(root_dir + 'workflow/scripts/')

from scanpy_init_env import *
from scanpy_utils import *
from scanpy_gene_lists import *

### Load data

In [ ]:
# Initialize the environment and get all paths and logger
logger, root_dir, sheets_dir, plate_path, scanpy_dir, report_dir = initialize_env(plate)
logger.info("Loading data ...")
adata = sc.read(scanpy_dir + f'adata_clusters_v3.h5ad')
adata.shape

### Dimensions

In [ ]:
# Dimensions
logger.info(f"Object dimensions (cells x genes): {adata.shape}")   

### Assign cell types

In [ ]:
logger.info("Setting cluster IDs ...")
# Create a new column in adata.obs with cell type names
adata.obs['cell_type'] = adata.obs['leiden_harmony_0.2'].map(cluster_anns)

# Check the new column
print(adata.obs[['leiden_harmony_0.2', 'cell_type']].head())

### Annotations

In [ ]:
# Plot feature plots res harmony 1.0
logger.info("Plotting UMAP and feature plots palette A ...")

fig = plot_celltype_and_gene_features(
    adata=adata,
    final_genes=final_genes,
    umap_palette=custom_palette,
    cell_type_column='cell_type',
    ncols=3,
    figsize_width=10.5,
    base_height_per_row=1.0
)
plt.show()

### Cell counts

In [ ]:
(
    adata.obs['cell_type']
    .value_counts()
    .reset_index()
    .rename(columns={"index": "cell_type"})
    .to_csv(report_dir + "scanpy_L1_cell_type_counts.tsv", sep="\t", index=False)
)
adata.obs['cell_type'].value_counts()

### Key Metrics

In [ ]:
# Total number of nuclei (assuming one per cell)
logger.info(f"Total number of nuclei: {adata.n_obs}")

# Median number of nuclei per sample
logger.info(f"Median nuclei per sample: {adata.obs['sample'].value_counts().median()}")

# Median number of reads per cell (assuming adata.X is count matrix; handles sparse/dense)
logger.info(f"Median reads per cell: {np.median(adata.X.sum(axis=1).A1)}") 

### Percentage of cells per sample per cluster

In [ ]:
plot_stacked_figure(adata, sample_column = "sample", color_column = "cell_type", barplot = True, recalculate_columns = True)